In [ ]:
# 📊 EDA Notebook — READY FOR GEMINI'S CODE
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/InfraNova-AI')

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import random

print("✅ EDA workspace ready!")
print(f"📂 Dataset location: data/raw/sentinel2/")
print(f"📊 Train samples: {len(os.listdir('data/raw/sentinel2/train/ir'))}")
print(f"📊 Val samples: {len(os.listdir('data/raw/sentinel2/val/ir'))}")
print(f"📊 Test samples: {len(os.listdir('data/raw/sentinel2/test/ir'))}")

# 🛰️ InfraNova AI: Infrared to RGB Satellite Image Colorization
## Phase 1: Exploratory Data Analysis (EDA) & Quality Assurance
**Dataset:** Sentinel-2 (Band 8 NIR to Bands 4,3,2 RGB)
**Resolution:** 64x64 (Native)
**Goal:** Verify dataset integrity, extract pixel distribution statistics, and generate publication-quality visual assets for the pitch deck.

In [ ]:
import os
import glob
import hashlib
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import defaultdict

# Pitch deck aesthetics
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

BASE_PATH = "/content/drive/MyDrive/InfraNova-AI/data/raw/sentinel2"
OUTPUT_DIR = "outputs/visualizations"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SPLITS = ['train', 'val', 'test']
MODALITIES = ['ir', 'rgb']

### 1. Dataset Statistics & 3. Pairing Verification
Validating directory structures, matching IR-RGB pairs, and computing core tensor statistics.

In [ ]:
dataset_info = {}
all_ir_pixels = []
all_rgb_pixels = []
orphan_files = []

for split in SPLITS:
    ir_path = os.path.join(BASE_PATH, split, 'ir')
    rgb_path = os.path.join(BASE_PATH, split, 'rgb')

    # Check if paths exist
    if not os.path.exists(ir_path) or not os.path.exists(rgb_path):
        print(f"⚠️ Warning: Missing directories for split '{split}'")
        continue

    ir_files = set([os.path.basename(f) for f in glob.glob(f"{ir_path}/*.*")])
    rgb_files = set([os.path.basename(f) for f in glob.glob(f"{rgb_path}/*.*")])

    # 3. Pairing Verification
    orphans_ir = ir_files - rgb_files
    orphans_rgb = rgb_files - ir_files
    if orphans_ir: orphan_files.extend([f"{split}/ir/{f}" for f in orphans_ir])
    if orphans_rgb: orphan_files.extend([f"{split}/rgb/{f}" for f in orphans_rgb])

    valid_pairs = list(ir_files & rgb_files)
    dataset_info[split] = valid_pairs

    print(f"[{split.upper()}] Valid Pairs: {len(valid_pairs)} | Orphans: {len(orphans_ir) + len(orphans_rgb)}")

    # 1. Pixel Statistics Calculation (Sampling train set to save memory)
    if split == 'train':
        print("Extracting pixel distributions from train set...")
        for f in tqdm(valid_pairs[:1000]): # Cap at 1000 for speed, sufficient for stats
            ir_img = np.array(Image.open(os.path.join(ir_path, f)).convert('L'))
            rgb_img = np.array(Image.open(os.path.join(rgb_path, f)).convert('RGB'))
            all_ir_pixels.append(ir_img.flatten())
            all_rgb_pixels.append(rgb_img.reshape(-1, 3))

if orphan_files:
    print(f"🚨 CRITICAL: Found {len(orphan_files)} orphan files. Ex: {orphan_files[:5]}")
else:
    print("✅ PERFECT MATCH: Every IR image has a corresponding RGB image.")

# Compute global stats
ir_arr = np.concatenate(all_ir_pixels)
rgb_arr = np.concatenate(all_rgb_pixels)

print("\n--- PIXEL INTENSITY STATISTICS (0-255) ---")
print(f"IR  -> Mean: {ir_arr.mean():.2f}, Std: {ir_arr.std():.2f}, Min: {ir_arr.min()}, Max: {ir_arr.max()}")
print(f"RGB -> Mean: {rgb_arr.mean(axis=0)}, Std: {rgb_arr.std(axis=0)}, Min: {rgb_arr.min(axis=0)}, Max: {rgb_arr.max(axis=0)}")

### 2. Sample Visualizations
Generating publication-quality grid of pairs for the hackathon pitch deck.

In [ ]:
def plot_samples(split='train', num_samples=8):
    pairs = dataset_info.get(split, [])
    if not pairs: return

    selected_files = np.random.choice(pairs, num_samples, replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    fig.suptitle('Sentinel-2 Multispectral Pairs (IR vs RGB)', fontsize=16, y=0.95, weight='bold')

    for i, file in enumerate(selected_files):
        # Determine grid position
        row_group = (i // 4) * 2  # 0 for first 4, 2 for next 4
        col = i % 4

        ir_img = Image.open(os.path.join(BASE_PATH, split, 'ir', file)).convert('L')
        rgb_img = Image.open(os.path.join(BASE_PATH, split, 'rgb', file)).convert('RGB')

        # Plot IR
        ax_ir = axes[row_group, col]
        ax_ir.imshow(ir_img, cmap='gray')
        ax_ir.set_title(f"IR: {file[:10]}", fontsize=8)
        ax_ir.axis('off')

        # Plot RGB
        ax_rgb = axes[row_group + 1, col]
        ax_rgb.imshow(rgb_img)
        ax_rgb.set_title(f"RGB: {file[:10]}", fontsize=8)
        ax_rgb.axis('off')

    plt.tight_layout()
    out_path = os.path.join(OUTPUT_DIR, "eda_samples.png")
    plt.savefig(out_path, bbox_inches='tight')
    plt.show()
    print(f"✅ Saved sample visualization to {out_path}")

plot_samples()

### 4. Resolution & 6. Potential Issues Report
Scanning the entire dataset for resolution mismatches, pure black/white artifacts, and exact duplicates.

In [ ]:
total_size_kb = 0
resolutions = set()
corrupt_files = []
black_white_files = []
hashes = set()
duplicates = 0

print("Scanning for anomalies and computing sizes...")
for split in SPLITS:
    for f in tqdm(dataset_info.get(split, [])):
        for mod in MODALITIES:
            path = os.path.join(BASE_PATH, split, mod, f)
            size_kb = os.path.getsize(path) / 1024.0
            total_size_kb += size_kb

            try:
                with Image.open(path) as img:
                    resolutions.add(img.size)
                    extrema = img.convert("L").getextrema()
                    # Check for pure black or pure white images
                    if extrema == (0, 0) or extrema == (255, 255):
                        black_white_files.append(path)

                    # Check duplicates via MD5 hash of raw data
                    img_hash = hashlib.md5(img.tobytes()).hexdigest()
                    if img_hash in hashes:
                        duplicates += 1
                    else:
                        hashes.add(img_hash)
            except Exception as e:
                corrupt_files.append(path)

print("\n--- RESOLUTION & SIZE ANALYSIS ---")
print(f"Unique Resolutions found: {resolutions} (Should be strictly {{(64, 64)}})")
print(f"Average File Size: {total_size_kb / (len(hashes) + duplicates):.2f} KB")
print(f"Total Dataset Size: {(total_size_kb / 1024):.2f} MB")

print("\n--- POTENTIAL ISSUES REPORT ---")
print(f"Corrupted Files: {len(corrupt_files)}")
print(f"Pure Black/White Artifacts: {len(black_white_files)}")
print(f"Exact Duplicates Detected: {duplicates // 2} pairs") # Divide by 2 because modalities share duplicates
if black_white_files: print(f"Sample artifacts: {black_white_files[:3]}")

### 5. Color & Histogram Distribution Analysis
Checking for spatial class biases (e.g., vegetation dominance).

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# IR Histogram
ax1.hist(np.random.choice(ir_arr, 100000), bins=50, color='gray', alpha=0.7)
ax1.set_title("IR Pixel Intensity Distribution")
ax1.set_xlabel("Pixel Value (0-255)")
ax1.set_ylabel("Frequency")

# RGB Color Distribution
rgb_sample = rgb_arr[np.random.choice(rgb_arr.shape[0], 100000, replace=False)]
ax2.hist(rgb_sample[:, 0], bins=50, color='red', alpha=0.5, label='Red')
ax2.hist(rgb_sample[:, 1], bins=50, color='green', alpha=0.5, label='Green')
ax2.hist(rgb_sample[:, 2], bins=50, color='blue', alpha=0.5, label='Blue')
ax2.set_title("RGB Channel Distribution")
ax2.set_xlabel("Pixel Value (0-255)")
ax2.legend()

plt.tight_layout()
out_path_hist = os.path.join(OUTPUT_DIR, "eda_histograms.png")
plt.savefig(out_path_hist, bbox_inches='tight')
plt.show()
print(f"✅ Saved histograms to {out_path_hist}")

### 🧠 7. KEY INSIGHTS FOR MODEL TRAINING (RESEARCH LEAD VERDICT)

Based on the EDA outputs, here is the immediate strategy for our pipeline:

1. **Class Bias & Color Jittering:**
   * *Observation:* Sentinel-2 Band 8 (NIR) is highly sensitive to chlorophyll. If the Green histogram heavily dominates the Red/Blue channels, our dataset is over-indexing on rural/forest topography.
   * *Action:* Apply aggressive `ColorJitter` (brightness, contrast, hue) **ONLY to the RGB targets** during training. This forces the generator to rely on IR texture rather than memorizing that "IR intensity X always maps to Green".

2. **Normalization Protocol:**
   * *Observation:* Our minimums and maximums span the full 0-255 range.
   * *Action:* Use standard GAN normalization mapping to `[-1, 1]` via `transforms.Normalize((0.5,), (0.5,))` for IR and `transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))` for RGB. This matches the `Tanh` activation at the end of the Pix2Pix/ThermalGAN generator.

3. **Upsampling Strategy (64x64 to 256x256):**
   * *Action:* Do NOT upscale the dataset on disk. Keep it raw. Implement the upscaling *inside* the PyTorch Dataset class using `Bicubic` interpolation. Bilinear causes checkerboard artifacts in generators, and Nearest Neighbor introduces harsh pixelation that damages structural loss (SSIM).

4. **Data Cleaning:**
   * *Action:* Any image paths printed under `POTENTIAL ISSUES REPORT` (pure black/white files) MUST be added to a `blacklist.txt` and skipped in the PyTorch `__getitem__` method to prevent the discriminator from destabilizing.

5. **Batch Size:**
   * *Action:* Given the low total size (~MBs) and 64x64 base resolution, we can push batch sizes higher to stabilize Batch Normalization layers. Recommend starting with a batch size of `16` or `32` on the Colab T4.